In [23]:
import numpy as np
import pandas as pd

In [24]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [25]:
df = pd.read_csv('covid_toy.csv')

In [26]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [56]:
df["gender"].value_counts()

gender
Female    59
Male      41
Name: count, dtype: int64

In [27]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train , y_test = train_test_split(df.drop(columns=["has_covid"]), df["has_covid"], test_size=0.2)

In [28]:
#In normal we do all the columns spearately 
#fever has 10 missing values 
#city and gender are categorial(Nominal data we use ohe)
#Cough is ordinal data in which we use ordinal encoding

In [29]:
simple = SimpleImputer()
X_train_fever = simple.fit_transform(X_train[["fever"]])
simple.transform(X_test[["fever"]]).shape

(20, 1)

In [30]:
oe = OrdinalEncoder(categories=[["Mild", "Strong"]])
X_train_cough = oe.fit_transform(X_train[["cough"]])

In [31]:
from sklearn.compose import ColumnTransformer


In [39]:
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'city'])
], remainder='passthrough')
#passthrough allows other columns other than transfromed to be as it is not dropping them

In [46]:
X_trained_transform = transformer.fit_transform(X_train)
X_trained_transform

array([[101.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  68.        ],
       [ 99.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  49.        ],
       [102.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,   5.        ],
       [104.        ,   0.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  17.        ],
       [ 98.        ,   1.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  71.        ],
       [103.        ,   1.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  46.        ],
       [100.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  19.        ],
       [104.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  12.        ],
       [ 99.        ,   1.        ,   0.        ,   1.        ,
          0.    

In [44]:
transformer.transform(X_test)

array([[100.91780822,   1.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  34.        ],
       [100.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  55.        ],
       [100.91780822,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  42.        ],
       [103.        ,   0.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  73.        ],
       [102.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  82.        ],
       [102.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   1.        ,  74.        ],
       [103.        ,   1.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  70.        ],
       [ 99.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  14.        ],
       [ 99.        ,   1.        ,   1.        ,   0.        ,
          0.    

In [47]:
df_transformed = pd.DataFrame(X_trained_transform, columns = transformer.get_feature_names_out())
df_transformed

,tnf1__fever,tnf2__cough,tnf3__gender_Male,tnf3__city_Delhi,tnf3__city_Kolkata,tnf3__city_Mumbai,remainder__age
0,101.0,1.0,0.0,1.0,0.0,0.0,68.0
1,99.0,1.0,0.0,0.0,0.0,0.0,49.0
2,102.0,0.0,1.0,0.0,1.0,0.0,5.0
3,104.0,0.0,0.0,0.0,1.0,0.0,17.0
4,98.0,1.0,0.0,0.0,1.0,0.0,71.0
...,...,...,...,...,...,...,...
75,104.0,0.0,1.0,0.0,1.0,0.0,16.0
76,103.0,0.0,0.0,0.0,1.0,0.0,69.0
77,98.0,1.0,1.0,0.0,0.0,0.0,12.0
78,101.0,1.0,1.0,0.0,0.0,0.0,47.0


Here , In Tnf3 we have done onehot encoding in city and gender such that it has decreased one column which is the first column which decreases the dimension so that multicollinearity is removed